# 08: The Combined Momentum Book

This notebook wraps up the whole project. It tells the story of how we went from a working
pair-trading system to a **two-leg, momentum-rotated book** — the instability we uncovered along the
way, the dead ends we ruled out, and the final configuration with its results next to S&P 500
buy-and-hold.


---

## 1. The arc of the investigation

Every step of this project was driven by a *failure* that forced the next experiment. The short
version: the original system earned decent returns, then we found it was fragile to the start date,
blamed earnings gaps, ruled that out, and discovered the real problem was pair-selection stability.
That led to the literature, a wider universe and 12-month selection, which was better — but it still
had no significant alpha. So we combined the two best legs into a momentum-rotated book.

| Step | What we did | Key result | Decision it produced |
|---|---|---|---|
| **1. A working baseline** | 2m selection, core, **5 pairs** (where the mp=5 grid left off) | ~2% annualized, Sharpe ≈ 1.3 on the recent window | A workable starting point |
| **2. …but start-date fragile** | 5 start dates × 7 folds | Sharpe swings −0.28 → +1.48 across 2-week start shifts; losses cluster around earnings gaps | Suspect the earnings screen |
| **3. Earnings screen** | Forward + backward earnings blackout windows | Inconsistent (14 better / 17 worse); blocks too many legit trades | Earnings isn't the driver |
| **4. Selection is the real problem** | Quantified selection stability across start dates | Jaccard@5 ≈ 0.04, dropout ≈ 94%; same-sector pool only ~20 pairs | **Root cause = tiny pool / selection volatility** |
| **5. Look to the literature** | Move to the full SP500 universe; grid 2m vs 12m selection (notebook 07) | SP500 2m is too noisy (best 0.74, std 0.82); **12m-selection flips it** | SP500 12m `cross 1m noscreen` is the strongest cell in the study |
| **6. Regimes reverse** | Push the winners to 2015–2019 | Cross-sector 3m wins historically; same-sector 1m collapses | What works recent ≠ what works historical |
| **7. But no significant alpha** | Fama-French on the book (notebook 09) | Alpha insignificant everywhere; **ST_Rev significant** | The edge is factor exposure, not alpha |
| **8. Combine the two best legs** | Momentum-rotated two-leg book | Final lb84 / step 0.40 / bounds 0.10–0.90 → **#1/#1/#1 in both mechanisms** | Lock in `clean40` |


---

## 2. Why a combined book

The final step makes the leap: instead of *picking one config* that has to be good in every regime,
deploy **two legs simultaneously** — each strong in its own window, not terrible in the other — and
let a **monthly momentum rotation** tilt capital toward whichever leg has been stronger recently.

- **Leg A (strong-recent):** `sp500-12m / cross_sector_slide1m_noscreen` — the strongest cell in the
  whole study on 2024–2025 (notebook 07).
- **Leg B (historical):** `sp500-2m / cross_sector_slide3m_bd7` — the historical-strong leg in
  the selected consensus pair.

The two legs are complementary *by construction*: one carries the recent regime, the other carries
the historical regime. The rotation decides how much of each we hold over time.


---

## 3. How we chose the pair

We swept all **496 pairs** of the 32 single-leg configs and ranked them three ways, because a single
metric is easy to fool:

1. **Min-score** — `min(recent Sharpe, hist Sharpe)`; rewards being strong in its regime *and* not
   terrible in the other.
2. **Rank-average** — mean of the pair's rank across the two windows.
3. **Joined** — single Sharpe of the concatenated recent ++ historical series.

The single-method winner (the bd7 variant) was **superseded** by a stricter
**3-method × 2-mechanism consensus**: a pair had to be in the **top-20 of all three ranking methods,
in BOTH capital-allocation mechanisms**. The corrected intersection produced exactly **7 surviving
pairs** — and our final pair ranked **#1 in all three methods in both mechanisms** across all 496
pairs.

**Final pair (locked):** `sp500-2m/cross_sector_slide3m_bd7` (leg B) + `sp500-12m/cross_sector_slide1m_noscreen`
(leg A).


---

## 4. Rotation mechanics & the weight-lattice trap

The allocation starts **50/50** and compares the trailing **84-day Sharpe** of each leg at the
start of each month. It shifts the weight on leg A by **±0.40** toward the winner and clamps it
to **[0.10, 0.90]**. Mechanism A re-bases active fold capital monthly; mechanism B locks an
unweighted leg-specific fold basis at fold start and applies the current weight only to new entry
flow. Open trades are not forcibly rebalanced.

### The weight-lattice trap (structural finding)

The rotation weight lives on the lattice `w0 ± k·step`. If a bound is **not on that lattice** — e.g.
bounds 0.15–0.85 at step 0.30, where 0.15 is 0.35 away from w0 — then clamping snaps the weight onto
a **second lattice that can never return to 0.50**. The book becomes permanently biased.

Exhaustive reachability check: `{0.15, 0.25, 0.45, 0.55, 0.75, 0.85}` is a **closed loop** — once the
weight touches a bound it can never come back to 0.50. **Only bounds = 0.50 ± integer·step are
clean.** This invalidated the default 0.25–0.75 (once clamped at 0.25 it oscillates on the odd
lattice) and every candidate with off-lattice bounds.

Our final config sits on the clean lattice **{0.10, 0.30, 0.50, 0.70, 0.90}**.

---

## 5. The final config

**Momentum rotation: lookback 84 / step 0.40 / bounds 0.10–0.90 / start w_A = 0.50.**

Every config was re-ranked against all **496 pairs** (not just our pair), so the choice is by *rank*,
not raw Sharpe:

| Config | Mech A (rec / hist) | Mech A ranks | Mech B (rec / hist) | Mech B ranks |
|---|---|---|---|---|
| **s0.40 / b0.10–0.90 (final)** | 1.094 / 1.122 | **#1 / #1 / #1** | 1.141 / 1.096 | **#1 / #1 / #1** |
| s0.50 / b0.00–1.00 (all-in) | 1.148 / 1.139 | #1 / #1 / #1 | 1.217 / 1.115 | #1 / #1 / #1 |
| s0.30 / b0.20–0.80 | 1.019 / 1.091 | #1 / #7 / #1 | 1.041 / 1.062 | #1 / #1 / #1 |
| default lb63 / s0.10 / b25–75 | 0.826 / 1.139 | #1 / #17 / #1 | 0.817 / 1.094 | #2 / #16 / #1 |

**Why clean40 and not the all-in 0–1 cell:** the all-in config has the best recent and historical
B Sharpe in this small comparison (1.217 and 1.115), but it can concentrate 100% in one leg.
Clean40 keeps a **10% diversification floor** while retaining the #1/#1/#1 rank result:

| Window | Mech | default Sh | tuned Sh | delta |
|---|---|---:|---:|---:|
| historical | A | 1.139 | 1.122 | **-0.017** |
| historical | B | 1.094 | 1.096 | **+0.003** |
| recent | A | 0.826 | 1.094 | **+0.268** |
| recent | B | 0.817 | 1.141 | **+0.324** |

**A/B are now exposure-comparable:** the B sizing correction uses a leg-specific active-fold
denominator, so its first-entry gross deployment matches A at equal weights. B is slightly better
recently and slightly worse historically under clean40; there is no universal mechanism winner.
A monthly re-bases its fold capital, while B lets its entry-flow allocation drift.


---

## 6. Results vs S&P 500 buy-and-hold

Annualized figures from the daily return series (recent = mean over five aligned starts;
historical = one aligned 2015–2019 start):

| Window | Series | Sharpe | Ann. return | Ann. vol |
|---|---:|---:|---:|---:|
| Historical | clean40 mech A | **1.12** | +9.1% | 8.0% |
| | clean40 mech B | **1.10** | +8.8% | 8.0% |
| | S&P 500 buy & hold | 0.87 | +11.9% | 13.7% |
| Recent | clean40 mech A | **1.09** | +9.4% | 8.6% |
| | clean40 mech B | **1.14** | +9.8% | 8.5% |
| | S&P 500 buy & hold | 1.21 | +20.4% | 16.9% |

**The book beats S&P on Sharpe historically** (1.10–1.12 vs 0.87), but trails it recently
(1.09–1.14 vs 1.21) while running about half the volatility. S&P 500 earns substantially more
absolute return in both windows. The honest framing is a **lower-vol diversifier**, not an S&P
substitute.

### Recent equity curve (2024–2025)

<img src="../fixed_diagnosis/clean40/pair_equity_recent.png" width="1000">

*Leg A (solid blue) drives recent performance; the pair mechanisms (green = A, orange dashed = B)
smooth it and lag S&P only in the strong melt-up periods.*

### Historical equity curve (2015–2019)

<img src="../fixed_diagnosis/clean40/pair_equity_historical.png" width="1000">

*Historical window: both mechanisms out-run S&P 500 on a risk-adjusted basis despite far lower
absolute growth — the book's low-vol, low-correlation profile.*

### The rotation at work

<img src="../fixed_diagnosis/clean40/weightA_recent.png" width="1000">

*Weight on leg A over the recent window. The 84-day lookback + 0.40 step + 0.10–0.90 bounds keep the
allocation responsive to recent strength while never fully abandoning either leg.*


---

## 7. Honest bottom line & limitations

**What this is:** a low-volatility, low-correlation **diversifier** that beats S&P 500 buy-and-hold
on Sharpe historically and remains lower-volatility recently, with the final tuned config ranked
**#1/#1/#1 (score, rank-average, joined) in both mechanisms** across all 496 pairs.

**What this is not:** an alpha engine. The Fama-French work (notebook 09) shows the book is
effectively a **short-term-reversal strategy with modest, statistically insignificant alpha** — the
edge is **factor exposure (ST_Rev), not unexplained return**. Tuning did not create alpha; it made
the book more robust.

**Limitations to keep front-of-mind:**

1. **In-sample selection.** The winning pair, the rotation rule, and the tuned parameters were all
   **selected on the same windows they're scored on**. Nothing here is out-of-sample.
2. **Trade-event-combined PnL.** The combined book replays existing leg trade events and daily marks
   into one shared account (1M start); cross-sleeve slippage, financing and transaction costs are
   not simulated.
3. **Start-date sensitivity.** The original instability was reduced but never fully eliminated;
   mid-month starts remain weaker than month-start starts.
4. **The value is structural, not proven alpha** — low vol, low correlation, and recent/historical
   regime differences; expect meaningful drawdown if short-term reversal stops being harvested.
5. **Mechanism semantics.** A monthly re-bases fold capital; B locks a leg-specific fold basis and
   applies the current weight only to new entries. They are exposure-comparable after the B sizing
   correction, but they are not identical strategies.
